# What's In My Thattu — Food Recognition Model Training

Trains a food recognition model on the **Indian Food Images** dataset (80 classes)
using EfficientNetV2B0 transfer learning, then exports a `.tflite` for the Android app.

## Before You Run — Add This Dataset

Click **"+ Add Input"** (right sidebar) and search for:

| Dataset | Kaggle Path | Classes |
|---------|-------------|---------|
| **Indian Food Images** | `iamsouravbanerjee/indian-food-images-dataset` | 80 |

> **Why just one dataset?** Merging 5 datasets creates 400+ classes with very few
> images each — the model can't learn. Starting with 80 well-labeled Indian food
> classes gives much better accuracy (expect 60-80%+). You can add more datasets
> later once this baseline is solid.

## Settings Required
- **Accelerator**: GPU T4 x2 (Settings → Accelerator)
- **Internet**: ON (Settings → Internet)
- **Persistence**: Files only (recommended)

## 1. Environment Setup

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tflite-support"])

import tensorflow as tf
import numpy as np
import os, shutil, json, glob, random
from pathlib import Path
from collections import defaultdict

# Reproducibility
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

print(f"TensorFlow: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)
print(f"GPUs: {len(gpus)} available" if gpus else "WARNING: No GPU! Enable in Settings -> Accelerator")

## 2. Configuration

In [ ]:
import os

# === Paths ===
INPUT_DIR = "/kaggle/input"
OUTPUT_DIR = "/kaggle/working"
DATASET_DIR = None  # Will be auto-detected
TFLITE_MODEL_PATH = os.path.join(OUTPUT_DIR, "whats_in_my_thattu_v2.tflite")
LABEL_MAP_PATH = os.path.join(OUTPUT_DIR, "labels.txt")
CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# === Image ===
IMAGE_SIZE = 224
CHANNELS = 3

# === Training (tuned for ~80 classes with ~100-200 images each) ===
BATCH_SIZE = 32
PHASE1_EPOCHS = 30          # Transfer learning (frozen base) — more epochs for head
PHASE2_EPOCHS = 20          # Fine-tuning (unfrozen top layers)
PHASE1_LR = 1e-3
PHASE2_LR = 1e-5
FINE_TUNE_AT_LAYER = 100    # Unfreeze layers after this index
VALIDATION_SPLIT = 0.20     # 20% for validation (more reliable metrics)
MIN_IMAGES_PER_CLASS = 10   # Need at least 10 images to learn a class

# === Regularization ===
DROPOUT_RATE = 0.4           # Slightly higher dropout for small dataset
LABEL_SMOOTHING = 0.1
WEIGHT_DECAY = 1e-4          # Stronger regularization

# === Callbacks ===
EARLY_STOPPING_PATIENCE = 7  # More patience — let it converge
REDUCE_LR_PATIENCE = 3
REDUCE_LR_FACTOR = 0.5
MIN_LR = 1e-7

print("Config loaded.")
print(f"  Phase 1: {PHASE1_EPOCHS} epochs @ LR={PHASE1_LR} (frozen base)")
print(f"  Phase 2: {PHASE2_EPOCHS} epochs @ LR={PHASE2_LR} (fine-tune)")
print(f"  Validation split: {VALIDATION_SPLIT*100:.0f}%")
print(f"  Min images per class: {MIN_IMAGES_PER_CLASS}")

## 3. Find the Dataset

Scans `/kaggle/input/` to find the Indian Food Images dataset.
Walks through the directory tree to locate the folder containing class subdirectories.

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def is_image(path):
    return Path(path).suffix.lower() in IMAGE_EXTENSIONS


def find_dataset_root(base_path, min_classes=10):
    """
    Walk the input directory tree to find the folder that contains
    class subdirectories (each with images inside).
    Returns (dataset_root, class_info_dict).
    """
    for root, dirs, files in os.walk(base_path):
        # Check if this directory's children are class folders with images
        class_info = {}
        for d in sorted(dirs):
            class_dir = os.path.join(root, d)
            img_count = sum(
                1 for f in os.listdir(class_dir)
                if os.path.isfile(os.path.join(class_dir, f)) and is_image(os.path.join(class_dir, f))
            )
            if img_count > 0:
                class_info[d] = img_count

        if len(class_info) >= min_classes:
            return root, class_info

    return None, {}


# Find the dataset
print(f"Scanning {INPUT_DIR} for food image dataset...")
DATASET_DIR, class_image_count_raw = find_dataset_root(INPUT_DIR)

if DATASET_DIR is None:
    raise RuntimeError(
        "No dataset found! Make sure you added 'Indian Food Images' via '+ Add Input'."
    )

print(f"\nDataset found: {DATASET_DIR}")
print(f"  Classes detected: {len(class_image_count_raw)}")
print(f"  Total images: {sum(class_image_count_raw.values()):,}")

# Filter out classes with too few images
class_image_count = {
    k: v for k, v in class_image_count_raw.items() if v >= MIN_IMAGES_PER_CLASS
}
removed = len(class_image_count_raw) - len(class_image_count)
if removed > 0:
    print(f"  Removed {removed} classes with < {MIN_IMAGES_PER_CLASS} images")

class_names = sorted(class_image_count.keys())
NUM_CLASSES = len(class_names)
total_images = sum(class_image_count.values())

print(f"\n{'='*50}")
print(f"DATASET SUMMARY")
print(f"{'='*50}")
print(f"  Classes:  {NUM_CLASSES}")
print(f"  Images:   {total_images:,}")
counts = sorted(class_image_count.values())
print(f"  Per class: min={counts[0]}, median={counts[len(counts)//2]}, max={counts[-1]}")
print(f"\nSample classes:")
for name in class_names[:10]:
    print(f"  {name:30s} ({class_image_count[name]} images)")
if NUM_CLASSES > 10:
    print(f"  ... and {NUM_CLASSES - 10} more")

## 4. Save Label Map

In [ ]:
# Save label map for TFLite metadata
with open(LABEL_MAP_PATH, "w") as f:
    for name in class_names:
        # Class folder names are already human-readable in Indian Food Images dataset
        display = name.replace("_", " ").strip().title()
        f.write(f"{display}\n")

print(f"Label map saved: {LABEL_MAP_PATH} ({NUM_CLASSES} classes)")
print(f"\nAll {NUM_CLASSES} classes:")
for i, name in enumerate(class_names):
    count = class_image_count[name]
    display = name.replace("_", " ").strip().title()
    print(f"  {i+1:3d}. {display:30s} ({count} images)")

## 5. Build Data Pipeline

Loads directly from the dataset directory using `image_dataset_from_directory`.
Applies aggressive data augmentation to compensate for the small dataset size
(~100-200 images per class). Class weights handle imbalanced classes.

In [ ]:
# --- Load datasets from dataset directory ---

print(f"Loading datasets from: {DATASET_DIR}")

# Training set (with validation split)
raw_train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=VALIDATION_SPLIT,
    subset="training",
    seed=SEED,
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=None,  # We'll batch after augmentation
    label_mode="int",
    shuffle=True,
)

# Validation set
raw_val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=VALIDATION_SPLIT,
    subset="validation",
    seed=SEED,
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=None,
    label_mode="int",
    shuffle=False,  # Don't shuffle validation for consistent eval
)

# Verify class count
loaded_class_names = raw_train_ds.class_names
print(f"  Loaded classes: {len(loaded_class_names)}")

# Count actual train/val samples
train_count = raw_train_ds.cardinality().numpy()
val_count = raw_val_ds.cardinality().numpy()
print(f"  Train samples: {train_count}")
print(f"  Val samples:   {val_count}")

In [ ]:
# --- Compute class weights to handle imbalanced classes ---

# Use loaded class names for consistency with the data pipeline
NUM_CLASSES = len(loaded_class_names)
total_for_weights = sum(class_image_count.get(n, 50) for n in loaded_class_names)

class_weights = {}
for i, name in enumerate(loaded_class_names):
    count = class_image_count.get(name, 50)  # fallback
    weight = total_for_weights / (NUM_CLASSES * count)
    class_weights[i] = min(max(weight, 0.5), 5.0)  # Clamp [0.5, 5.0]

weights_list = [class_weights[i] for i in range(NUM_CLASSES)]
print(f"Class weights computed for {NUM_CLASSES} classes:")
print(f"  min={min(weights_list):.2f}, max={max(weights_list):.2f}, "
      f"median={sorted(weights_list)[len(weights_list)//2]:.2f}")

In [ ]:
# --- Preprocessing & augmentation functions ---

AUTOTUNE = tf.data.AUTOTUNE

def normalize(image, label):
    """Normalize pixels to [0, 1]."""
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

def one_hot(image, label):
    """One-hot encode the label."""
    return image, tf.one_hot(label, NUM_CLASSES)

# Aggressive data augmentation for small dataset (~100-200 images/class)
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.15),           # ±15% rotation
    tf.keras.layers.RandomZoom((-0.2, 0.0)),         # Up to 20% zoom in
    tf.keras.layers.RandomTranslation(0.1, 0.1),     # ±10% shift
    tf.keras.layers.RandomBrightness(0.2),            # ±20% brightness
    tf.keras.layers.RandomContrast(0.2),              # ±20% contrast
], name="data_augmentation")

def augment(image, label):
    """Apply augmentation to a single image."""
    image = data_augmentation(image, training=True)
    return image, label

# --- Build final pipelines ---

train_ds = (
    raw_train_ds
    .shuffle(8000, seed=SEED)
    .map(normalize, num_parallel_calls=AUTOTUNE)
    .map(augment, num_parallel_calls=AUTOTUNE)
    .map(one_hot, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

val_ds = (
    raw_val_ds
    .map(normalize, num_parallel_calls=AUTOTUNE)
    .map(one_hot, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

train_batches = tf.data.experimental.cardinality(train_ds).numpy()
val_batches = tf.data.experimental.cardinality(val_ds).numpy()
print(f"Pipeline ready:")
print(f"  Train batches: {train_batches} ({train_batches * BATCH_SIZE} samples/epoch)")
print(f"  Val batches:   {val_batches}")

## 6. Visualize Sample Training Images

In [ ]:
import matplotlib.pyplot as plt

sample_images, sample_labels = next(iter(train_ds))

fig, axes = plt.subplots(3, 5, figsize=(16, 10))
for i, ax in enumerate(axes.flat):
    if i < len(sample_images):
        ax.imshow(sample_images[i].numpy().clip(0, 1))
        label_idx = tf.argmax(sample_labels[i]).numpy()
        name = loaded_class_names[label_idx].replace("_", " ").title()
        ax.set_title(name, fontsize=9)
    ax.axis("off")

plt.suptitle(f"Sample Training Images — {NUM_CLASSES} Classes (with augmentation)", fontsize=14)
plt.tight_layout()
plt.show()

## 7. Build the Model

EfficientNetV2B0 (pretrained ImageNet) + custom classification head.
- Stronger regularization (dropout 0.4, L2 weight decay) for small dataset
- Two dense layers with batch normalization for feature refinement

In [ ]:
from tensorflow.keras import layers, regularizers

def build_model(num_classes):
    """Build food classifier with EfficientNetV2B0 backbone."""
    inputs = tf.keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, CHANNELS))
    
    base_model = tf.keras.applications.EfficientNetV2B0(
        include_top=False,
        weights="imagenet",
        input_tensor=inputs,
        include_preprocessing=False,
    )
    base_model.trainable = False
    
    x = base_model.output
    x = layers.GlobalAveragePooling2D(name="global_avg_pool")(x)
    x = layers.BatchNormalization(name="bn_1")(x)
    x = layers.Dense(512, activation="relu",
                     kernel_regularizer=regularizers.l2(WEIGHT_DECAY),
                     name="dense_1")(x)
    x = layers.Dropout(DROPOUT_RATE, name="dropout_1")(x)
    x = layers.BatchNormalization(name="bn_2")(x)
    x = layers.Dense(256, activation="relu",
                     kernel_regularizer=regularizers.l2(WEIGHT_DECAY),
                     name="dense_2")(x)
    x = layers.Dropout(DROPOUT_RATE, name="dropout_2")(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="predictions")(x)
    
    model = tf.keras.Model(inputs=inputs, outputs=outputs, name="food_classifier_v2")
    return model, base_model

model, base_model = build_model(NUM_CLASSES)

trainable = sum(tf.keras.backend.count_params(w) for w in model.trainable_weights)
total = model.count_params()
print(f"Model built for {NUM_CLASSES} food classes")
print(f"  Total params:     {total:,}")
print(f"  Trainable params: {trainable:,} (head only)")
print(f"  Base model frozen: {not base_model.trainable}")

## 8. Phase 1 — Transfer Learning (Frozen Base)

Train only the classification head with frozen EfficientNet backbone.
With 80 classes this should reach **40-60% val accuracy** in Phase 1.

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=PHASE1_LR),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
    metrics=[
        "accuracy",
        tf.keras.metrics.TopKCategoricalAccuracy(k=5, name="top5_accuracy"),
    ],
)

phase1_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=os.path.join(CHECKPOINT_DIR, "best_phase1.keras"),
        monitor="val_accuracy", mode="max",
        save_best_only=True, verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=EARLY_STOPPING_PATIENCE,
        mode="max", restore_best_weights=True, verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=REDUCE_LR_FACTOR,
        patience=REDUCE_LR_PATIENCE, min_lr=MIN_LR, verbose=1,
    ),
]

print("=" * 60)
print(f"PHASE 1: Transfer Learning — {NUM_CLASSES} classes, frozen base")
print("=" * 60)

history_p1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=PHASE1_EPOCHS,
    class_weight=class_weights,
    callbacks=phase1_callbacks,
)

print(f"\nPhase 1 complete!")
print(f"  Best val accuracy: {max(history_p1.history['val_accuracy'])*100:.2f}%")
print(f"  Best val top-5:    {max(history_p1.history['val_top5_accuracy'])*100:.2f}%")

## 9. Phase 2 — Fine-Tuning (Unfreeze Top Layers)

Unfreeze the top ~130 layers of EfficientNet for domain adaptation.
With fine-tuning this should reach **60-80%+ val accuracy**.

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:FINE_TUNE_AT_LAYER]:
    layer.trainable = False

unfrozen = sum(1 for l in base_model.layers if l.trainable)
frozen = sum(1 for l in base_model.layers if not l.trainable)
print(f"Fine-tuning: {unfrozen} unfrozen, {frozen} frozen layers")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=PHASE2_LR),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
    metrics=[
        "accuracy",
        tf.keras.metrics.TopKCategoricalAccuracy(k=5, name="top5_accuracy"),
    ],
)

phase2_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=os.path.join(CHECKPOINT_DIR, "best_phase2.keras"),
        monitor="val_accuracy", mode="max",
        save_best_only=True, verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=EARLY_STOPPING_PATIENCE,
        mode="max", restore_best_weights=True, verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=REDUCE_LR_FACTOR,
        patience=REDUCE_LR_PATIENCE, min_lr=MIN_LR, verbose=1,
    ),
]

print("\n" + "=" * 60)
print("PHASE 2: Fine-Tuning (top layers unfrozen)")
print("=" * 60)

history_p2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=PHASE2_EPOCHS,
    class_weight=class_weights,
    callbacks=phase2_callbacks,
)

print(f"\nPhase 2 complete!")
print(f"  Best val accuracy: {max(history_p2.history['val_accuracy'])*100:.2f}%")
print(f"  Best val top-5:    {max(history_p2.history['val_top5_accuracy'])*100:.2f}%")

## 10. Training History Plots

In [ ]:
import matplotlib.pyplot as plt

def plot_history(h1, h2):
    acc = h1.history["accuracy"] + h2.history["accuracy"]
    val_acc = h1.history["val_accuracy"] + h2.history["val_accuracy"]
    loss = h1.history["loss"] + h2.history["loss"]
    val_loss = h1.history["val_loss"] + h2.history["val_loss"]
    top5 = h1.history["top5_accuracy"] + h2.history["top5_accuracy"]
    val_top5 = h1.history["val_top5_accuracy"] + h2.history["val_top5_accuracy"]
    
    epochs = range(1, len(acc) + 1)
    phase1_end = len(h1.history["accuracy"])
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    for ax, (train, val, title) in zip(axes, [
        (acc, val_acc, "Top-1 Accuracy"),
        (top5, val_top5, "Top-5 Accuracy"),
        (loss, val_loss, "Loss"),
    ]):
        ax.plot(epochs, train, "b-", label="Train", linewidth=2)
        ax.plot(epochs, val, "r-", label="Validation", linewidth=2)
        ax.axvline(x=phase1_end, color="gray", linestyle="--", alpha=0.7, label="Fine-tune start")
        ax.set_title(title, fontsize=13)
        ax.set_xlabel("Epoch")
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    plt.suptitle(f"Training History — {NUM_CLASSES} Indian Food Classes", fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "training_curves.png"), dpi=150, bbox_inches="tight")
    plt.show()

plot_history(history_p1, history_p2)

## 11. Evaluate on Validation Set

In [ ]:
print("Evaluating on validation set...")
val_results = model.evaluate(val_ds, verbose=1)

print(f"\n{'='*50}")
print(f"VALIDATION RESULTS ({NUM_CLASSES} classes)")
print(f"{'='*50}")
print(f"  Loss:           {val_results[0]:.4f}")
print(f"  Top-1 Accuracy: {val_results[1]*100:.2f}%")
print(f"  Top-5 Accuracy: {val_results[2]*100:.2f}%")

## 12. Per-Class Analysis

In [ ]:
from collections import Counter

all_preds = []
all_labels = []

for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    all_preds.append(preds)
    all_labels.append(labels.numpy())

all_preds = np.concatenate(all_preds, axis=0)
all_labels = np.concatenate(all_labels, axis=0)

pred_classes = np.argmax(all_preds, axis=1)
true_classes = np.argmax(all_labels, axis=1)

per_class = []
for i, name in enumerate(loaded_class_names):
    mask = true_classes == i
    if mask.sum() == 0:
        continue
    acc = np.mean(pred_classes[mask] == i)
    display = name.replace("_", " ").title()
    per_class.append((display, acc, int(mask.sum())))

per_class.sort(key=lambda x: x[1])

print("WORST 15 classes:")
print("-" * 55)
for name, acc, count in per_class[:15]:
    bar = "█" * int(acc * 20) + "░" * (20 - int(acc * 20))
    print(f"  {name:30s} {bar} {acc*100:5.1f}% ({count})")

print(f"\nBEST 15 classes:")
print("-" * 55)
for name, acc, count in per_class[-15:]:
    bar = "█" * int(acc * 20) + "░" * (20 - int(acc * 20))
    print(f"  {name:30s} {bar} {acc*100:5.1f}% ({count})")

# Most confused pairs
confusion_pairs = Counter()
for t, p in zip(true_classes, pred_classes):
    if t != p:
        tn = loaded_class_names[t].replace("_", " ").title()
        pn = loaded_class_names[p].replace("_", " ").title()
        confusion_pairs[(tn, pn)] += 1

print(f"\nMOST CONFUSED PAIRS:")
print("-" * 65)
for (tn, pn), count in confusion_pairs.most_common(10):
    print(f"  {tn:28s} → {pn:28s} ({count}x)")

## 13. Export to TFLite

Float16 quantization — cuts model size ~50% with negligible accuracy loss.

In [ ]:
print("Converting to TFLite (float16 quantization)...")

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]

tflite_model = converter.convert()

with open(TFLITE_MODEL_PATH, "wb") as f:
    f.write(tflite_model)

size_mb = os.path.getsize(TFLITE_MODEL_PATH) / (1024 * 1024)
print(f"\nTFLite model saved: {TFLITE_MODEL_PATH}")
print(f"Model size: {size_mb:.1f} MB")
print(f"Classes: {NUM_CLASSES}")

In [ ]:
# --- Embed metadata + labels into the TFLite model ---

try:
    from tflite_support.metadata_writers import image_classifier
    from tflite_support.metadata_writers import writer_utils

    writer = image_classifier.MetadataWriter.create_for_inference(
        writer_utils.load_file(TFLITE_MODEL_PATH),
        model_name="What's In My Thattu Food Classifier v2",
        model_description=(
            f"Food recognition model with {NUM_CLASSES} categories. "
            f"Trained using EfficientNetV2B0 transfer learning on multiple "
            f"combined food datasets for broad coverage."
        ),
        input_norm_mean=[0.0],
        input_norm_std=[255.0],
        label_file_paths=[LABEL_MAP_PATH],
    )

    writer_utils.save_file(writer.populate(), TFLITE_MODEL_PATH)
    print("Metadata + labels embedded into TFLite model.")
    print("Android's mlModelBinding will auto-generate the wrapper class.")

except Exception as e:
    print(f"Metadata embedding skipped: {e}")
    print("Saving labels.txt separately as fallback.")
    import shutil
    fallback = TFLITE_MODEL_PATH.replace(".tflite", "_labels.txt")
    shutil.copy(LABEL_MAP_PATH, fallback)
    print(f"Fallback labels: {fallback}")

## 14. Verify TFLite Model

In [ ]:
print("Verifying TFLite model...")

interpreter = tf.lite.Interpreter(model_path=TFLITE_MODEL_PATH)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print(f"  Input:  shape={input_details[0]['shape']}, dtype={input_details[0]['dtype']}")
print(f"  Output: shape={output_details[0]['shape']}, dtype={output_details[0]['dtype']}")
assert output_details[0]['shape'][1] == NUM_CLASSES, \
    f"Output shape mismatch: expected {NUM_CLASSES}, got {output_details[0]['shape'][1]}"

# Test with a real validation image
test_batch = next(iter(val_ds))
test_image = test_batch[0][0:1].numpy().astype(np.float32)
true_label = np.argmax(test_batch[1][0].numpy())

interpreter.set_tensor(input_details[0]["index"], test_image)
interpreter.invoke()
output = interpreter.get_tensor(output_details[0]["index"])[0]

top5_indices = np.argsort(output)[::-1][:5]
true_name = loaded_class_names[true_label].replace("_", " ").title()

print(f"\n  True label: {true_name}")
print(f"  TFLite Top-5 predictions:")
for i, idx in enumerate(top5_indices, 1):
    name = loaded_class_names[idx].replace("_", " ").title()
    marker = " ✓" if idx == true_label else ""
    print(f"    {i}. {name:30s} {output[idx]*100:.1f}%{marker}")

print(f"\n  Softmax sum: {output.sum():.4f} (should be ~1.0)")
print(f"  Verification: {'PASSED ✓' if abs(output.sum() - 1.0) < 0.1 else 'WARNING: softmax sum off'}")

## 15. TFLite Accuracy Spot-Check

In [ ]:
NUM_EVAL = 500
correct = correct_top5 = total = 0

for images, labels in val_ds:
    for i in range(images.shape[0]):
        if total >= NUM_EVAL:
            break
        img = images[i:i+1].numpy().astype(np.float32)
        true_label = np.argmax(labels[i].numpy())
        
        interpreter.set_tensor(input_details[0]["index"], img)
        interpreter.invoke()
        output = interpreter.get_tensor(output_details[0]["index"])[0]
        
        if np.argmax(output) == true_label:
            correct += 1
        if true_label in np.argsort(output)[::-1][:5]:
            correct_top5 += 1
        total += 1
    if total >= NUM_EVAL:
        break

print(f"TFLite accuracy on {total} samples:")
print(f"  Top-1: {correct/total*100:.2f}%")
print(f"  Top-5: {correct_top5/total*100:.2f}%")

## 16. Save Everything & Summary

In [ ]:
# Save Keras model for future fine-tuning
keras_path = os.path.join(OUTPUT_DIR, "food_classifier_v2.keras")
model.save(keras_path)

# Save training report
report = {
    "num_classes": NUM_CLASSES,
    "class_names": [n.replace("_", " ").title() for n in loaded_class_names],
    "total_images": total_images,
    "phase1_best_val_acc": float(max(history_p1.history['val_accuracy'])),
    "phase2_best_val_acc": float(max(history_p2.history['val_accuracy'])),
    "val_top1_accuracy": float(val_results[1]),
    "val_top5_accuracy": float(val_results[2]),
    "tflite_size_mb": os.path.getsize(TFLITE_MODEL_PATH) / (1024 * 1024),
}
with open(os.path.join(OUTPUT_DIR, "training_report.json"), "w") as f:
    json.dump(report, f, indent=2)

tflite_mb = os.path.getsize(TFLITE_MODEL_PATH) / (1024 * 1024)
keras_mb = os.path.getsize(keras_path) / (1024 * 1024)

print("\n" + "=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)
print(f"  Food classes:       {NUM_CLASSES}")
print(f"  Total images:       {total_images:,}")
print(f"  Phase 1 best val:   {max(history_p1.history['val_accuracy'])*100:.2f}%")
print(f"  Phase 2 best val:   {max(history_p2.history['val_accuracy'])*100:.2f}%")
print(f"  Final val top-1:    {val_results[1]*100:.2f}%")
print(f"  Final val top-5:    {val_results[2]*100:.2f}%")
print(f"")
print(f"  OUTPUT FILES (download from Output tab):")
print(f"    whats_in_my_thattu_v2.tflite   ({tflite_mb:.1f} MB) ← Deploy to Android")
print(f"    labels.txt                      ({NUM_CLASSES} classes)")
print(f"    food_classifier_v2.keras        ({keras_mb:.1f} MB)")
print(f"    training_report.json")
print(f"    training_curves.png")
print(f"")
if val_results[1] < 0.4:
    print("  ⚠️  Accuracy below 40% — consider increasing PHASE1_EPOCHS or PHASE2_EPOCHS")
elif val_results[1] < 0.6:
    print("  📊 Decent accuracy — try adding more training epochs to improve further")
else:
    print("  ✅ Good accuracy — model is ready for deployment!")

## 17. Deploy to Android

### Step 1: Download
Download **both** files from the **Output** tab:
- `whats_in_my_thattu_v2.tflite`
- `labels.txt`

### Step 2: Copy to project
```bash
# Model file
cp whats_in_my_thattu_v2.tflite \
  tensorImageInterpreter/src/main/ml/whats_in_my_thattu_v2.tflite

# Labels file
cp labels.txt \
  tensorImageInterpreter/src/main/assets/labels.txt
```

### Step 3: Rebuild
The Android code is already configured — just rebuild the app.
The `TensorImageInterpreter` preprocesses images correctly (resize 224×224, normalize to [0,1]).

### Expected accuracy targets
| Metric | Good | Great |
|--------|------|-------|
| Top-1 | 50-65% | 65%+ |
| Top-5 | 80-90% | 90%+ |
| TFLite vs Keras gap | < 3% | < 1% |